# Compile post-AGB binary systems

Combines two sources:

### 1. Kluska et al. 2022
https://ui.adsabs.harvard.edu/abs/2022A%26A...658A..36K/abstract

_A population of transition disks around evolved stars: Fingerprints of planets._  
_Catalog of disks surrounding Galactic post-AGB binaries._  
Kluska, J.; Van Winckel, H.; Coppée, Q.; Oomen, G.-M.; Dsilva, K.; Kamath, D.; Bujarrabal, V.; Min, M.

Output: `Kluska2021_postAGB.raw.json` (85 systems)

### 2. Oomen et al. 2018
https://ui.adsabs.harvard.edu/abs/2018A%26A...620A..85O/abstract

_Post-AGB stars with hot dust and binarity as a tool for binary stellar evolution._

Output: `Oomen2018_postAGB.raw.json` (33 systems)

In [36]:
import numpy as np
import pandas as pd
import json
import re
import subprocess
import ast
import os
import sys
from pathlib import Path

# Ensure project root is on sys.path
proj_root = Path('/Users/liekevanson/Documents/Projects/post_mt_review').resolve()
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

from paths import DATA_DIR, RAW_JSON_DIR


---
## 1. Kluska et al. 2022

Parses coordinates, period, and eccentricity directly from the plain-text table file.

In [37]:
KLUSKA_INPUT = DATA_DIR / "from_others" / "Kluska_2021_table1.txt"
KLUSKA_OUTPUT = RAW_JSON_DIR / "Kluska2021_postAGB.raw.json"
REFERENCE_BIBCODE = "2022A&A...658A..36K"


def hms_to_deg(hms):
    parts = hms.split(":")
    if len(parts) != 3:
        return None
    try:
        h, m, s = map(float, parts)
        return 15.0 * (h + m / 60.0 + s / 3600.0)
    except ValueError:
        return None


def dms_to_deg(dms):
    parts = dms.split(":")
    if len(parts) != 3:
        return None
    try:
        sign = -1.0 if parts[0].startswith("-") else 1.0
        d = abs(float(parts[0]))
        m = float(parts[1])
        s = float(parts[2])
        return sign * (d + m / 60.0 + s / 3600.0)
    except ValueError:
        return None


def tokenize_with_blanks(raw_line):
    # Preserve blank second-name column by splitting on 2+ spaces.
    cols = [c.strip() for c in re.split(r"\s{2,}", raw_line.rstrip("\n"))]
    # Remove empty trailing chunks but keep internal blanks.
    while cols and cols[-1] == "":
        cols.pop()
    return cols


def all_float_tokens(text):
    return [float(x) for x in re.findall(r"[-+]?\d+(?:\.\d+)?", text)]


def parse_period_ecc_from_tail(tail_tokens):
    # Find "yes/no" marker and parse numerics preceding it.
    yn_idx = None
    for i, tok in enumerate(tail_tokens):
        if tok.lower() in {"yes", "no"}:
            yn_idx = i
            break

    parse_region = tail_tokens[:yn_idx] if yn_idx is not None else tail_tokens
    nums = []
    for tok in parse_region:
        nums.extend(all_float_tokens(tok))

    if not nums:
        return None, None
    if len(nums) == 1:
        return nums[-1], None

    # Typical table format has period then eccentricity as the final two numbers before yes/no.
    period_candidate = nums[-2]
    ecc_candidate = nums[-1]

    if 0.0 <= ecc_candidate <= 1.0 and period_candidate > 1.0:
        return period_candidate, ecc_candidate

    # If the final value is clearly not an eccentricity, keep only period.
    return nums[-1], None


def parse_kluska_line(raw_line):
    cols = tokenize_with_blanks(raw_line)
    if len(cols) < 5:
        return None

    # First non-empty column may be IRAS id (or blank for continuation rows).
    iras_id = cols[0] if cols[0] else None
    common_name = cols[1] if len(cols) > 1 and cols[1] else None

    # Coordinates are usually in cols[2], cols[3] with spaces; normalize to colon format.
    ra_raw = cols[2] if len(cols) > 2 else ""
    dec_raw = cols[3] if len(cols) > 3 else ""
    ra_hms = re.sub(r"\s+", ":", ra_raw.strip()) if ra_raw else ""
    dec_dms = re.sub(r"\s+", ":", dec_raw.strip()) if dec_raw else ""

    ra_deg = hms_to_deg(ra_hms)
    dec_deg = dms_to_deg(dec_dms)

    # Category appears near col[4] (e.g. Cat. 1 / Uncategorized)
    category_tag = cols[4] if len(cols) > 4 else ""

    # Remaining tokens hold stellar params + period/ecc + flags.
    tail_tokens = cols[5:] if len(cols) > 5 else []
    period_val, ecc_val = parse_period_ecc_from_tail(tail_tokens)

    names = [n for n in [iras_id, common_name] if n]
    if not names:
        return None

    system_name = names if len(names) > 1 else names[0]

    entry = {
        "System Name": system_name,
        "RA": [None, ra_deg, None],
        "Dec": [None, dec_deg, None],
        "Period": [None, period_val, None],
        "Eccentricity": [None, ecc_val, None],
        "M1": [None, None, None],
        "M2": [None, None, None],
        "Mass Function": [None, None, None],
        "M1_sin3i": [None, None, None],
        "M2_sin3i": [None, None, None],
        "evol_type_1": "MS",
        "evol_type_2": "None",
        "obs_type_1": "Post-AGB",
        "obs_type_2": "pre-WD?",
        "system_class": "Post-AGB binary",
        "Detection Method": ["RV"],
        "Reference": [REFERENCE_BIBCODE],
        "Notes": f"Source: Kluska+2021 table 1 ({category_tag}). Raw row: {raw_line.strip()}",
        "Simbad": None,
    }
    return entry


In [38]:
kluska_entries = []
with open(KLUSKA_INPUT, "r", encoding="utf-8") as fh:
    for raw in fh:
        raw = raw.rstrip("\n")
        stripped = raw.strip()
        if not stripped:
            continue
        # Skip the column-header line ("IRAS   Name ...") and the units line ("- - hh m ...")
        if re.match(r'^IRAS\s', stripped) or stripped.startswith("-"):
            continue
        entry = parse_kluska_line(raw)
        if entry is not None:
            kluska_entries.append(entry)

---
## 2. Oomen et al. 2018

Data hard-coded from the three LaTeX tables in the paper (orbital elements, mass functions, spectroscopic data).  
RA/Dec are queried from SIMBAD.

In [39]:
OOMEN_OUTPUT  = RAW_JSON_DIR / "Oomen2018_postAGB.raw.json"
OOMEN_BIBCODE = "2018A&A...620A..85O"

# ---------- Helper parsers ----------

def pm_to_triplet(value_str):
    """
    Convert 'value±err' strings to [err-, value, err+] triplets.
    Handles upper-only errors like '0.0+0.05' as [0.0, 0.0, 0.05].
    Returns None when the value is missing ('/').
    """
    s = value_str.strip()
    if s in ("/", "") or s.lower() == "nan":
        return None
    s = (s.replace("$", "").replace("\\pm", "±")
          .replace("\\,", "").replace("~", " ").replace("\\", ""))
    # upper-only uncertainty like "0.0+0.05"
    if "+" in s and "±" not in s:
        parts = s.split("+")
        try:
            val  = float(parts[0])
            errp = float(parts[1])
            return [0.0, val, errp]
        except Exception:
            pass
    # standard "value±err"
    if "±" in s:
        val_str, err_str = s.split("±")
        val = float(val_str)
        err = float(err_str)
        return [err, val, err]
    # plain number
    try:
        return [0.0, float(s), 0.0]
    except Exception:
        return None


def as_lower_limit_triplet(value):
    """Encode a lower limit as [0.0, value, +inf]."""
    return [0.0, float(value), float("inf")] if value is not None else None


TRI_NAN = [np.nan, np.nan, np.nan]


def safe_triplet(x):
    """Normalise a triplet-like value; returns NaN triplet if None."""
    if x is None:
        return TRI_NAN.copy()
    try:
        if len(x) == 3:
            return [float(x[0]), float(x[1]), float(x[2])]
    except Exception:
        pass
    try:
        return [np.nan, float(x), np.nan]
    except Exception:
        return TRI_NAN.copy()


columns = [
    "System Name", "RA", "Dec", "Period", "Eccentricity",
    "M1", "M1_sin3i", "M2", "M2_sin3i", "q", "Mass Function",
    "Type1", "Type2", "Detection Method", "Reference", "Notes"
]
oomen_df = pd.DataFrame(columns=columns)


def add_observation(df, system_name, ra, dec, period, ecc,
                    m1, m1_sin3i, m2, m2_sin3i, q, mass_func,
                    type1, type2, method, reference, notes=""):
    new_row = {
        "System Name": system_name, "RA": ra, "Dec": dec,
        "Period": period, "Eccentricity": ecc,
        "M1": m1, "M1_sin3i": m1_sin3i, "M2": m2, "M2_sin3i": m2_sin3i,
        "q": q, "Mass Function": mass_func,
        "Type1": type1, "Type2": type2,
        "Detection Method": method, "Reference": reference, "Notes": notes,
    }
    return pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

In [40]:
# ---------- Table 1: orbital elements ----------
tab1_rows = [
    (1,  "89 Her",          "289.1±0.2",    "0.29±0.07",   "1"),
    (2,  "AC Her",          "1188.9±1.2",   "0.0+0.05",    "2"),
    (3,  "BD+39 4926",      "871.7±0.4",    "0.024±0.006", "3"),
    (4,  "BD+46 442",       "140.82±0.02",  "0.085±0.005", "4"),
    (5,  "DY Ori",          "1248±36",      "0.22±0.08",   "5"),
    (6,  "EP Lyr",          "1151±14",      "0.39±0.09",   "5"),
    (7,  "HD 44179",        "317.6±1.1",    "0.27±0.03",   "6,7"),
    (8,  "HD 46703",        "597.4±0.2",    "0.30±0.02",   "8"),
    (9,  "HD 52961",        "1288.6±0.3",   "0.23±0.01",   "7"),
    (10, "HD 95767",        "1989±61",      "0.25±0.05",   "9"),
    (11, "HD 108015",       "906.3±5.9",    "0.0+0.03",    "9"),
    (12, "HD 131356",       "1488.0±8.7",   "0.32±0.04",   "9"),
    (13, "HD 158616",       "363.3±1.0",    "0.0+0.1",     "10"),
    (14, "HD 213985",       "259.6±0.7",    "0.21±0.05",   "9"),
    (15, "HP Lyr",          "1818±80",      "0.20±0.04",   "5"),
    (16, "HR 4049",         "430.6±0.1",    "0.30±0.01",   "11"),
    (17, "IRAS 05208-2035", "234.38±0.04",  "0.0+0.02",    "17"),
    (18, "IRAS 06165+3158", "262.6±0.7",    "0.0+0.05",    ""),
    (19, "IRAS 06452-3456", "215.4±0.4",    "0.0+0.03",    ""),
    (20, "IRAS 08544-4431", "501.1±1.0",    "0.20±0.02",   "12"),
    (21, "IRAS 09144-4933", "1762±27",      "0.30±0.04",   "5"),
    (22, "IRAS 15469-5311", "390.2±0.7",    "0.08±0.02",   "12"),
    (23, "IRAS 16230-3410", "649.8±3.5",    "0.0+0.13",    ""),
    (24, "IRAS 17038-4815", "1394±12",      "0.63±0.06",   "5"),
    (25, "IRAS 19125+0343", "519.7±0.7",    "0.24±0.03",   "12"),
    (26, "IRAS 19135+3937", "126.97±0.08",  "0.13±0.03",   "13"),
    (27, "IRAS 19157-0247", "119.6±0.1",    "0.34±0.04",   "12"),
    (28, "RU Cen",          "1489±10",      "0.62±0.07",   "14"),
    (29, "SAO 173329",      "115.951±0.002","0.0+0.04",    "9"),
    (30, "ST Pup",          "406.0±2.2",    "0.0+0.04",    "15"),
    (31, "SX Cen",          "564.3±7.6",    "0.0+0.06",    "14"),
    (32, "TW Cam",          "662.2±5.3",    "0.25±0.04",   "5"),
    (33, "U Mon",           "2550±143",     "0.25±0.06",   "16"),
]
tab1 = {name: {"Period": pm_to_triplet(per), "Eccentricity": pm_to_triplet(ecc)}
        for _, name, per, ecc, _ in tab1_rows}

# ---------- Table 2: mass functions and minimum companion masses ----------
tab2_rows = [
    (1,  "89 Her",          "0.106±0.007",  "0.0019±0.0004",  "0.10"),
    (2,  "AC Her",          "1.176±0.080",  "0.153±0.032",    "0.64"),
    (3,  "BD+39 4926",      "1.286±0.007",  "0.373±0.006",    "1.03"),
    (4,  "BD+46 442",       "0.3074±0.0014","0.195±0.003",    "0.72"),
    (5,  "DY Ori",          "1.39±0.11",    "0.23±0.05",      "0.79"),
    (6,  "EP Lyr",          "1.30±0.12",    "0.22±0.06",      "0.77"),
    (7,  "HD 44179",        "0.342±0.008",  "0.053±0.003",    "0.38"),
    (8,  "HD 46703",        "0.839±0.015",  "0.220±0.012",    "0.77"),
    (9,  "HD 52961",        "1.507±0.034",  "0.274±0.019",    "0.87"),
    (10, "HD 95767",        "2.14±0.16",    "0.33±0.07",      "0.96"),
    (11, "HD 108015",       "0.28±0.02",    "0.0036±0.0009",  "0.13"),
    (12, "HD 131356",       "2.11±0.09",    "0.57±0.07",      "1.33"),
    (13, "HD 158616",       "0.28±0.03",    "0.022±0.008",    "0.26"),
    (14, "HD 213985",       "0.733±0.025",  "0.777±0.079",    "1.62"),
    (15, "HP Lyr",          "1.27±0.06",    "0.083±0.007",    "0.47"),
    (16, "HR 4049",         "0.627±0.010",  "0.177±0.008",    "0.69"),
    (17, "IRAS 05208-2035", "0.396±0.004",  "0.150±0.005",    "0.64"),
    (18, "IRAS 06165+3158", "0.374±0.011",  "0.10±0.01",      "0.52"),
    (19, "IRAS 06452-3456", "0.73±0.01",    "1.12±0.05",      "2.07"),
    (20, "IRAS 08544-4431", "0.398±0.008",  "0.033±0.002",    "0.31"),
    (21, "IRAS 09144-4933", "2.25±0.11",    "0.49±0.07",      "1.21"),
    (22, "IRAS 15469-5311", "0.438±0.015",  "0.074±0.008",    "0.45"),
    (23, "IRAS 16230-3410", "0.232±0.021",  "0.004±0.001",    "0.13"),
    (24, "IRAS 17038-4815", "1.52±0.08",    "0.24±0.04",      "0.81"),
    (25, "IRAS 19125+0343", "0.56±0.02",    "0.086±0.010",    "0.48"),
    (26, "IRAS 19135+3937", "0.209±0.008",  "0.075±0.008",    "0.45"),
    (27, "IRAS 19157-0247", "0.083±0.003",  "0.0053±0.0007",  "0.15"),
    (28, "RU Cen",          "2.38±0.15",    "0.81±0.17",      "1.66"),
    (29, "SAO 173329",      "0.132±0.003",  "0.023±0.001",    "0.27"),
    (30, "ST Pup",          "0.67±0.02",    "0.241±0.026",    "0.81"),
    (31, "SX Cen",          "1.12±0.05",    "0.58±0.07",      "1.35"),
    (32, "TW Cam",          "0.83±0.04",    "0.174±0.022",    "0.68"),
    (33, "U Mon",           "3.38±0.31",    "0.79±0.18",      "1.64"),
]
tab2 = {name: {"Mass Function": pm_to_triplet(f), "M2_min": float(m2min)}
        for _, name, _, f, m2min in tab2_rows}

# ---------- Table 3: spectroscopic / depletion data (used for Notes only) ----------
tab3_rows = [
    (1,  "89 Her",          "6600", "0.8",  "0.02", "0.38",  "-0.5", "mild"),
    (2,  "AC Her",          "5800", "1.0",  "0.46", "0.24",  "-1.4", "mild"),
    (3,  "BD+39 4926",      "7750", "1.0",  "0.23", "0.0",   "-2.4", "strong"),
    (4,  "BD+46 442",       "6250", "1.5",  "0.23", "0.19",  "-0.8", "no"),
    (5,  "DY Ori",          "5900", "1.5",  "0.90", "0.74",  "-2.3", "strong"),
    (6,  "EP Lyr",          "6200", "1.5",  "0.48", "0.04",  "-1.8", "moderate"),
    (7,  "HD 44179",        "7500", "0.8",  "0.15", "18.1",  "-3.3", "strong"),
    (8,  "HD 46703",        "6250", "1.0",  "0.23", "0.02",  "-1.7", "mild"),
    (9,  "HD 52961",        "6000", "0.5",  "0.04", "0.13",  "-4.8", "strong"),
    (10, "HD 95767",        "7500", "2.0",  "0.58", "0.55",  "0.1",  "no"),
    (11, "HD 108015",       "7000", "1.5",  "0.15", "1.04",  "-0.1", "no"),
    (12, "HD 131356",       "6000", "1.0",  "0.15", "0.65",  "0.0",  "mild"),
    (13, "HD 158616",       "7250", "1.25", "0.51", "0.23",  "-0.6", "no"),
    (14, "HD 213985",       "8250", "1.5",  "0.12", "0.35",  "-0.9", "strong"),
    (15, "HP Lyr",          "6300", "1.0",  "0.39", "0.56",  "-1.0", "strong"),
    (16, "HR 4049",         "7600", "1.1",  "0.20", "0.12",  "-4.8", "strong"),
    (17, "IRAS 05208-2035", "4250", "0.75", "0.01", "0.43",  "-0.7", "no"),
    (18, "IRAS 06165+3158", "4250", "1.5",  "0.53", "0.39",  "-0.9", "no"),
    (19, "IRAS 06452-3456", "/",   "/",    "0.94", "0.11",  "/",    "/"),
    (20, "IRAS 08544-4431", "7250", "1.5",  "1.32", "0.49",  "-0.3", "mild"),
    (21, "IRAS 09144-4933", "5750", "0.5",  "1.78", "0.81",  "-0.3", "moderate"),
    (22, "IRAS 15469-5311", "7500", "1.5",  "1.27", "0.74",  "0.0",  "strong"),
    (23, "IRAS 16230-3410", "6250", "1.0",  "0.72", "0.46",  "-0.7", "moderate"),
    (24, "IRAS 17038-4815", "4750", "0.5",  "0.57", "0.79",  "-1.5", "no"),
    (25, "IRAS 19125+0343", "7750", "1.0",  "0.94", "0.90",  "-0.3", "strong"),
    (26, "IRAS 19135+3937", "6000", "0.5",  "0.28", "0.26",  "-1.0", "no"),
    (27, "IRAS 19157-0247", "7750", "1.0",  "0.66", "0.79",  "0.1",  "no"),
    (28, "RU Cen",          "6000", "1.5",  "0.18", "0.39",  "-1.9", "moderate"),
    (29, "SAO 173329",      "7000", "1.0",  "0.31", "0.35",  "-0.9", "no"),
    (30, "ST Pup",          "5500", "1.0",  "0.06", "1.32",  "-1.5", "strong"),
    (31, "SX Cen",          "6250", "1.5",  "0.17", "0.40",  "-1.1", "strong"),
    (32, "TW Cam",          "4800", "0.0",  "0.42", "0.43",  "-0.5", "no"),
    (33, "U Mon",           "5000", "0.0",  "0.34", "0.28",  "-0.8", "no"),
]
tab3 = {
    name: {"EBV": None if ebv == "/" else float(ebv),
           "LIR": None if lir == "/" else float(lir),
           "Depletion": depl}
    for _, name, _teff, _logg, ebv, lir, _feh, depl in tab3_rows
}

# ---------- Build the DataFrame ----------
for name in [r[1] for r in tab1_rows]:
    per  = safe_triplet(tab1[name]["Period"])
    ecc  = safe_triplet(tab1[name]["Eccentricity"])
    mf   = safe_triplet(tab2[name]["Mass Function"])
    m2ll = as_lower_limit_triplet(tab2[name]["M2_min"])

    notes_bits = []
    if name in tab3:
        ebv  = tab3[name]["EBV"]
        lir  = tab3[name]["LIR"]
        depl = tab3[name]["Depletion"]
        if ebv  is not None:         notes_bits.append(f"E(B-V)={ebv}")
        if lir  is not None:         notes_bits.append(f"L_IR/L_*={lir}")
        if depl and depl != "/":    notes_bits.append(f"Depletion={depl}")
    notes_bits.append("Companion minimum mass assumes M1=0.6 Msun and i=75 deg (from Table 2).")
    notes = "; ".join(notes_bits)

    oomen_df = add_observation(
        oomen_df,
        system_name=name,
        ra=TRI_NAN.copy(),
        dec=TRI_NAN.copy(),
        period=per, ecc=ecc,
        m1=TRI_NAN.copy(), m1_sin3i=TRI_NAN.copy(),
        m2=m2ll, m2_sin3i=TRI_NAN.copy(),
        q=TRI_NAN.copy(), mass_func=mf,
        type1="MS?", type2="post AGB",
        method=["RV"],
        reference=[OOMEN_BIBCODE],
        notes=notes,
    )

print(f"Built Oomen DataFrame: {len(oomen_df)} rows")
display(oomen_df[["System Name", "Period", "Eccentricity", "Mass Function", "M2"]])

Built Oomen DataFrame: 33 rows


,System Name,Period,Eccentricity,Mass Function,M2
0,89 Her,"[0.2, 289.1, 0.2]","[0.07, 0.29, 0.07]","[0.0004, 0.0019, 0.0004]","[0.0, 0.1, inf]"
1,AC Her,"[1.2, 1188.9, 1.2]","[0.0, 0.0, 0.05]","[0.032, 0.153, 0.032]","[0.0, 0.64, inf]"
2,BD+39 4926,"[0.4, 871.7, 0.4]","[0.006, 0.024, 0.006]","[0.006, 0.373, 0.006]","[0.0, 1.03, inf]"
3,BD+46 442,"[0.02, 140.82, 0.02]","[0.005, 0.085, 0.005]","[0.003, 0.195, 0.003]","[0.0, 0.72, inf]"
4,DY Ori,"[36.0, 1248.0, 36.0]","[0.08, 0.22, 0.08]","[0.05, 0.23, 0.05]","[0.0, 0.79, inf]"
5,EP Lyr,"[14.0, 1151.0, 14.0]","[0.09, 0.39, 0.09]","[0.06, 0.22, 0.06]","[0.0, 0.77, inf]"
6,HD 44179,"[1.1, 317.6, 1.1]","[0.03, 0.27, 0.03]","[0.003, 0.053, 0.003]","[0.0, 0.38, inf]"
7,HD 46703,"[0.2, 597.4, 0.2]","[0.02, 0.3, 0.02]","[0.012, 0.22, 0.012]","[0.0, 0.77, inf]"
8,HD 52961,"[0.3, 1288.6, 0.3]","[0.01, 0.23, 0.01]","[0.019, 0.274, 0.019]","[0.0, 0.87, inf]"
9,HD 95767,"[61.0, 1989.0, 61.0]","[0.05, 0.25, 0.05]","[0.07, 0.33, 0.07]","[0.0, 0.96, inf]"


In [41]:
# ---------- Query SIMBAD for RA/Dec ----------
targets = oomen_df["System Name"].tolist()

result = subprocess.run(
    ["python3", "Get_Coords_From_SIMBAD.py"] + targets,
    capture_output=True, text=True
)

RADEC_data = {}
for name, dict_str in re.findall(r"([A-Z0-9_]+)\s*=\s*({.*?})", result.stdout, re.DOTALL):
    RADEC_data[name] = ast.literal_eval(dict_str)

RA_values  = [RADEC_data[k]["RA"]  for k in RADEC_data]
DEC_values = [RADEC_data[k]["Dec"] for k in RADEC_data]

oomen_df["RA"]  = RA_values
oomen_df["Dec"] = DEC_values

# Use the single Oomen bibcode for all rows
oomen_df["Reference"] = [[OOMEN_BIBCODE]] * len(oomen_df)

print(f"RA/Dec populated for {len(RA_values)} systems")
display(oomen_df)

RA/Dec populated for 33 systems


,System Name,RA,Dec,Period,Eccentricity,M1,M1_sin3i,M2,M2_sin3i,q,Mass Function,Type1,Type2,Detection Method,Reference,Notes
0,89 Her,"[1.144e-05, 268.854951, 1.144e-05]","[1.275e-05, 26.049991, 1.275e-05]","[0.2, 289.1, 0.2]","[0.07, 0.29, 0.07]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.1, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.0004, 0.0019, 0.0004]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.02; L_IR/L_*=0.38; Depletion=mild; Co...
1,AC Her,"[4.13e-06, 277.567655, 4.13e-06]","[7.5e-06, 21.866833, 7.5e-06]","[1.2, 1188.9, 1.2]","[0.0, 0.0, 0.05]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.64, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.032, 0.153, 0.032]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.46; L_IR/L_*=0.24; Depletion=mild; Co...
2,BD+39 4926,"[8.03e-06, 341.54678, 8.03e-06]","[6.53e-06, 40.107308, 6.53e-06]","[0.4, 871.7, 0.4]","[0.006, 0.024, 0.006]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 1.03, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.006, 0.373, 0.006]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.23; L_IR/L_*=0.0; Depletion=strong; C...
3,BD+46 442,"[6.86e-06, 26.445963, 6.86e-06]","[3.42e-06, 46.816927, 3.42e-06]","[0.02, 140.82, 0.02]","[0.005, 0.085, 0.005]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.72, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.003, 0.195, 0.003]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.23; L_IR/L_*=0.19; Depletion=no; Comp...
4,DY Ori,"[2.203e-05, 91.562119, 2.203e-05]","[1.956e-05, 13.905311, 1.956e-05]","[36.0, 1248.0, 36.0]","[0.08, 0.22, 0.08]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.79, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.05, 0.23, 0.05]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.9; L_IR/L_*=0.74; Depletion=strong; C...
5,EP Lyr,"[4.52e-06, 289.581488, 4.52e-06]","[5.19e-06, 27.850856, 5.19e-06]","[14.0, 1151.0, 14.0]","[0.09, 0.39, 0.09]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.77, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.06, 0.22, 0.06]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.48; L_IR/L_*=0.04; Depletion=moderate...
6,HD 44179,"[0.00050026, 94.992577, 0.00050026]","[0.00040556, -10.637418, 0.00040556]","[1.1, 317.6, 1.1]","[0.03, 0.27, 0.03]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.38, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.003, 0.053, 0.003]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.15; L_IR/L_*=18.1; Depletion=strong; ...
7,HD 46703,"[1.018e-05, 99.468445, 1.018e-05]","[5.78e-06, 53.517211, 5.78e-06]","[0.2, 597.4, 0.2]","[0.02, 0.3, 0.02]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.77, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.012, 0.22, 0.012]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.23; L_IR/L_*=0.02; Depletion=mild; Co...
8,HD 52961,"[7.49e-06, 105.915127, 7.49e-06]","[6.44e-06, 10.770297, 6.44e-06]","[0.3, 1288.6, 0.3]","[0.01, 0.23, 0.01]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.87, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.019, 0.274, 0.019]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.04; L_IR/L_*=0.13; Depletion=strong; ...
9,HD 95767,"[7.5e-06, 165.517981, 7.5e-06]","[3.72e-06, -62.161898, 3.72e-06]","[61.0, 1989.0, 61.0]","[0.05, 0.25, 0.05]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.96, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.07, 0.33, 0.07]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.58; L_IR/L_*=0.55; Depletion=no; Comp...


---
## 3. Moltzer et al. 2025

Compiled catalog of Galactic post-AGB and post-RGB binaries, parsed from `Moltzer2025.tex` (Table 1).  
Includes both spectroscopically and photometrically determined orbital periods.  
RA/Dec are not provided in the source table — set to `[None, None, None]`.

In [42]:
MOLTZER_INPUT   = DATA_DIR / "latex_input_data" / "Moltzer2025.tex"
MOLTZER_OUTPUT  = RAW_JSON_DIR / "Moltzer2025_postAGB.raw.json"

# Reference number → ADS bibcode (update placeholder strings as needed)
MOLTZER_REF_MAP = {
    "1":  "Csornyei2019",          # AU Peg
    "2":  "2018A&A...620A..85O",   # Oomen+2018
    "3":  "Bolton1980",            # HD 137569
    "4":  "Maas2003",              # HD 93662
    "5":  "Manick2019",            # RV Tau, HP Lyr, ...
    "6":  "Manick2021",            # V510 Pup
    "7":  "Kiss2007",              # photometric
    "8":  "2022A&A...658A..36K",   # Kluska+2022
    "9":  "Bodi2019",              # BT Lac
    "10": "Percy2015",             # BT Lac, R Sge
    "11": "Kiss2017",              # photometric
    "12": "Horne2012",             # RS Sge
    "13": "Hrivnak2024",           # QY Sge
}


def parse_moltzer_refs(ref_str):
    """Convert '(2)' or '(9)(10)' to a list of bibcodes."""
    nums = re.findall(r'\((\d+)\)', ref_str)
    return [MOLTZER_REF_MAP.get(n, f"ref_{n}") for n in nums]


def parse_pm_or_none(raw_str):
    """pm_to_triplet wrapper: returns [None, val, None] when no error is stated."""
    s = raw_str.strip()
    if not s or s == "...":
        return None
    has_err = ("\\pm" in s or "±" in s)
    tri = pm_to_triplet(s)
    if tri is None:
        return None
    return tri if has_err else [None, tri[1], None]


def parse_moltzer_tex(filepath):
    entries = []
    with open(filepath, "r", encoding="utf-8") as fh:
        lines = fh.readlines()

    for line in lines:
        line = line.strip()
        # Skip non-data lines
        if not line or line.count("&") != 4:
            continue
        if any(line.startswith(kw) for kw in (
            "\\hline", "\\multicolumn", "\\tablef", "\\tablebib",
            "\\end", "\\begin", "\\caption", "\\label", "\\centering",
        )):
            continue

        parts      = [p.strip() for p in line.split("&")]
        iras_raw   = parts[0]
        name_raw   = parts[1]
        period_raw = parts[2]
        ecc_raw    = parts[3]
        ref_raw    = parts[4].replace("\\\\", "").strip()

        # IRAS id (strip leading F, ignore "...")
        iras_id = None
        if iras_raw not in ("...", ""):
            iras_id = re.sub(r'^F', '', iras_raw.strip())

        # Alternative names: split on ";", drop "..."
        alt_names = [n.strip() for n in name_raw.split(";")
                     if n.strip() and n.strip() != "..."]

        # Period and eccentricity
        period = parse_pm_or_none(period_raw)
        is_photometric = ecc_raw.strip() in ("...", "")
        ecc    = None if is_photometric else parse_pm_or_none(ecc_raw)

        refs = parse_moltzer_refs(ref_raw)

        all_names = ([iras_id] if iras_id else []) + alt_names
        if not all_names:
            continue
        system_name = all_names if len(all_names) > 1 else all_names[0]

        entries.append({
            "System Name":      system_name,
            "RA":               [None, None, None],
            "Dec":              [None, None, None],
            "Period":           period if period else [None, None, None],
            "Eccentricity":     ecc    if ecc    else [None, None, None],
            "M1":               [None, None, None],
            "M2":               [None, None, None],
            "Mass Function":    [None, None, None],
            "M1_sin3i":         [None, None, None],
            "M2_sin3i":         [None, None, None],
            "evol_type_1":      "MS",
            "evol_type_2":      "None",
            "obs_type_1":       "Post-AGB",
            "obs_type_2":       "pre-WD?",
            "system_class":     "Post-AGB binary",
            "Detection Method": ["Photometric"] if is_photometric else ["RV"],
            "Reference":        refs,
            "Notes":            "Source: Moltzer+2025 Table 1.",
            "Simbad":           None,
        })
    return entries


moltzer_entries = parse_moltzer_tex(MOLTZER_INPUT)
print(f"Parsed {len(moltzer_entries)} Moltzer+2025 entries")
print(json.dumps(moltzer_entries[0], indent=2) if moltzer_entries else "No entries parsed")

Parsed 54 Moltzer+2025 entries
{
  "System Name": [
    "IRAS",
    "Name"
  ],
  "RA": [
    null,
    null,
    null
  ],
  "Dec": [
    null,
    null,
    null
  ],
  "Period": [
    null,
    null,
    null
  ],
  "Eccentricity": [
    null,
    null,
    null
  ],
  "M1": [
    null,
    null,
    null
  ],
  "M2": [
    null,
    null,
    null
  ],
  "Mass Function": [
    null,
    null,
    null
  ],
  "M1_sin3i": [
    null,
    null,
    null
  ],
  "M2_sin3i": [
    null,
    null,
    null
  ],
  "evol_type_1": "MS",
  "evol_type_2": "None",
  "obs_type_1": "Post-AGB",
  "obs_type_2": "pre-WD?",
  "system_class": "Post-AGB binary",
  "Detection Method": [
    "RV"
  ],
  "Reference": [],
  "Notes": "Source: Moltzer+2025 Table 1.",
  "Simbad": null
}


---
## 4. Export all three catalogs to raw JSON

In [43]:
def _triplet_to_json_value(tri):
    """Convert a triplet to [err-, value, err+] with JSON-safe nulls."""
    if tri is None:
        return [None, None, None]
    if not isinstance(tri, (list, tuple, np.ndarray)):
        try:
            v = float(tri)
            return [None, v if np.isfinite(v) else None, None]
        except Exception:
            return [None, None, None]
    out = []
    for x in list(tri)[:3]:
        try:
            xv = float(x)
            out.append(xv if np.isfinite(xv) else None)
        except Exception:
            out.append(None)
    while len(out) < 3:
        out.append(None)
    return out


def _listify(v):
    if v is None:
        return []
    if isinstance(v, list):
        return v
    if isinstance(v, tuple):
        return list(v)
    return [v]


def convert_oomen_df_to_entries(df):
    entries = []
    for _, row in df.iterrows():
        notes   = row.get("Notes", None)
        raw_m2  = row.get("M2", None)
        m2_tri  = _triplet_to_json_value(raw_m2)

        # +inf upper error means this is a lower limit — capture that in Notes
        try:
            if (isinstance(raw_m2, (list, tuple, np.ndarray)) and len(raw_m2) == 3
                    and not np.isfinite(float(raw_m2[2]))):
                extra = "M2 stored as lower limit from Oomen+2018 Table 2."
                notes = f"{notes}; {extra}" if notes else extra
        except Exception:
            pass

        entries.append({
            "System Name":    row.get("System Name", None),
            "RA":             _triplet_to_json_value(row.get("RA",           None)),
            "Dec":            _triplet_to_json_value(row.get("Dec",          None)),
            "Period":         _triplet_to_json_value(row.get("Period",       None)),
            "Eccentricity":   _triplet_to_json_value(row.get("Eccentricity", None)),
            "M1":             _triplet_to_json_value(row.get("M1",           None)),
            "M2":             m2_tri,
            "Mass Function":  _triplet_to_json_value(row.get("Mass Function",None)),
            "M1_sin3i":       _triplet_to_json_value(row.get("M1_sin3i",     None)),
            "M2_sin3i":       _triplet_to_json_value(row.get("M2_sin3i",     None)),
            "evol_type_1":    "MS",
            "evol_type_2":    "None",
            "obs_type_1":     "Post-AGB",
            "obs_type_2":     "pre-WD?",
            "system_class":   "Post-AGB binary",
            "Detection Method": _listify(row.get("Detection Method", [])),
            "Reference":      _listify(row.get("Reference",          [])),
            "Notes":          notes,
            "Simbad":         None,
        })
    return entries


def save_raw_json(entries, output_file):
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as fh:
        json.dump(entries, fh, separators=(",", ": "), ensure_ascii=False, indent=None)
    print(f"Saved {len(entries)} entries to {output_file}")


# # --- Save Kluska ---
# save_raw_json(kluska_entries, KLUSKA_OUTPUT)
# print(json.dumps(kluska_entries[0], indent=2) if kluska_entries else "")

# # --- Save Oomen ---
oomen_entries = convert_oomen_df_to_entries(oomen_df)
# save_raw_json(oomen_entries, OOMEN_OUTPUT)
# print(json.dumps(oomen_entries[0], indent=2) if oomen_entries else "")

# # --- Save Moltzer ---
# save_raw_json(moltzer_entries, MOLTZER_OUTPUT)
# print(json.dumps(moltzer_entries[0], indent=2) if moltzer_entries else "")

---
## 5. Cross-catalog overlap

Find systems that appear in two or more of the three catalogs by comparing normalized name variants.  
Uses union-find to group entries that share any name (IRAS id, common name, or alternative designation).

In [44]:
from collections import defaultdict

# --- Build catalogs dict and prepare for overlap detection ---
catalogs = {
    "Kluska": kluska_entries,
    "Oomen": oomen_entries,
    "Moltzer": moltzer_entries,
}

print(f"Catalog sizes: Kluska={len(kluska_entries)}, Oomen={len(oomen_entries)}, Moltzer={len(moltzer_entries)}")


def normalize_name(name):
    """Standardize a name to uppercase, remove IRAS prefix, strip punctuation."""
    if not name:
        return ""
    s = str(name).upper().strip()
    # Remove leading F from F-type IRAS
    s = re.sub(r'^F(?=IRAS)', '', s)
    # Remove punctuation and extra spaces
    s = re.sub(r'[\s._~;]', '', s)
    return s


def get_all_names(entry):
    """Extract all name variants from System Name (handles lists and strings)."""
    sn = entry.get("System Name")
    if not sn:
        return []
    if isinstance(sn, list):
        return [n for n in sn if n]
    if isinstance(sn, str):
        # Split on semicolon AND comma
        return [n.strip() for n in re.split(r'[;,]', sn) if n.strip()]
    return [str(sn)]


def extract_coords(entry):
    """Pull (RA, Dec) as floats from triplet format, handling None values."""
    ra = entry.get("RA", [None, None, None])
    dec = entry.get("Dec", [None, None, None])
    try:
        ra_val = ra[1] if isinstance(ra, list) and len(ra) > 1 else ra
        dec_val = dec[1] if isinstance(dec, list) and len(dec) > 1 else dec
        if ra_val is not None:
            ra_val = float(ra_val)
        if dec_val is not None:
            dec_val = float(dec_val)
        return (ra_val, dec_val) if ra_val is not None and dec_val is not None else None
    except (TypeError, ValueError):
        return None


def coords_match(c1, c2, tol_deg=1/3600):
    """Check if two (RA, Dec) coordinates match within tolerance (default 1 arcsec)."""
    if c1 is None or c2 is None:
        return False
    try:
        ra1, dec1 = float(c1[0]), float(c1[1])
        ra2, dec2 = float(c2[0]), float(c2[1])
        if not (np.isfinite(ra1) and np.isfinite(dec1) and np.isfinite(ra2) and np.isfinite(dec2)):
            return False
        dist = np.sqrt((ra1 - ra2)**2 + (dec1 - dec2)**2)
        return dist < tol_deg
    except (TypeError, ValueError):
        return False


# --- Union-find for grouping duplicates by name and coordinate matching ---
parent = {}
rank = {}

def uf_find(x):
    if x not in parent:
        parent[x] = x
        rank[x] = 0
    if parent[x] != x:
        parent[x] = uf_find(parent[x])
    return parent[x]

def uf_union(a, b):
    ra, rb = uf_find(a), uf_find(b)
    if ra == rb:
        return
    if rank[ra] < rank[rb]:
        parent[ra] = rb
    elif rank[ra] > rank[rb]:
        parent[rb] = ra
    else:
        parent[rb] = ra
        rank[ra] += 1

# --- Build all_keys and name_to_keys for single-pass union-find ---
all_keys = []
name_to_keys = defaultdict(list)

for src, entries_list in catalogs.items():
    for idx, entry in enumerate(entries_list):
        key = (src, idx)
        all_keys.append(key)
        for name in get_all_names(entry):
            norm = normalize_name(name)
            if norm:
                name_to_keys[norm].append(key)

# --- First pass: union by name match ---
for norm_name, keys_with_name in name_to_keys.items():
    for i in range(1, len(keys_with_name)):
        uf_union(keys_with_name[0], keys_with_name[i])

# --- Second pass: union by coordinate match (within same union-find group) ---
for i, key_i in enumerate(all_keys):
    coords_i = extract_coords(catalogs[key_i[0]][key_i[1]])
    if coords_i is None:
        continue
    for key_j in all_keys[i+1:]:
        coords_j = extract_coords(catalogs[key_j[0]][key_j[1]])
        if coords_j is None:
            continue
        if coords_match(coords_i, coords_j):
            uf_union(key_i, key_j)

print(f"Initialized union-find with {len(all_keys)} entries")


Catalog sizes: Kluska=85, Oomen=33, Moltzer=54
Initialized union-find with 172 entries


In [45]:

# Group entries by union-find root
groups_map = defaultdict(list)
for key in all_keys:
    root = uf_find(key)
    groups_map[root].append(key)

print(f"DEBUG: groups_map has {len(groups_map)} groups total")
print(f"DEBUG: Groups with >1 entry: {sum(1 for g in groups_map.values() if len(g) > 1)}")

# Keep only groups that span ≥2 catalogs
multi_groups = sorted(
    [g for g in groups_map.values() if len({src for src, _ in g}) >= 2],
    key=lambda g: min(catalogs[src][idx].get("Period", [None, None, None])[1] or 0
                      for src, idx in g)
)

DEBUG: groups_map has 96 groups total
DEBUG: Groups with >1 entry: 52


---
## 6. Merge into a single combined post AGB fiel

Merge all three sources into one file (`postAGB_combined.raw.json`).

**Field priority for overlapping systems:**
| Field | Priority |
|---|---|
| RA / Dec | Oomen (SIMBAD) → Kluska (table coords) → Moltzer (none) |
| Period / Eccentricity | Oomen (symmetric ±) → Kluska → Moltzer |
| Mass Function / M1 / M2 | Oomen only |
| System Name | union of all name variants |
| Reference | union of all bibcodes |
| Detection Method | prefer RV over Photometric |
| Notes | concatenated |

In [50]:
# --- Save as JSON array (compatible with Combine_and_process_data.py) ---
COMBINED_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with open(COMBINED_OUTPUT, "w", encoding="utf-8") as fh:
    fh.write("[\n")
    for i, system in enumerate(merged_entries):
        line = json.dumps(system, separators=(",", ": "), ensure_ascii=False)
        fh.write("  " + line)
        if i < len(merged_entries) - 1:
            fh.write(",\n")
        else:
            fh.write("\n")
    fh.write("]\n")

print(f"Saved {len(merged_entries)} entries to {COMBINED_OUTPUT}")

# --- Summary ---
n_duplicates = sum(1 for g in groups_map.values() if len(g) > 1)
n_unique   = len(merged_entries)
print(f"\n{n_duplicates} duplicate/overlap groups merged, {n_unique} total unique systems in combined catalog.")
print(f"\nFirst entry:\n{json.dumps(merged_entries[0], indent=2)}")

Saved 96 entries to /Users/liekevanson/Documents/Projects/post_mt_review/data/result_tables/raw_json/postAGB_combined.raw.json

52 duplicate/overlap groups merged, 96 total unique systems in combined catalog.

First entry:
{
  "System Name": "IRAS13110-5425",
  "RA": [
    null,
    198.53441666666666,
    null
  ],
  "Dec": [
    null,
    -54.692952777777776,
    null
  ],
  "Period": [
    null,
    2.0,
    null
  ],
  "Eccentricity": [
    null,
    null,
    null
  ],
  "M1": [
    null,
    null,
    null
  ],
  "M2": [
    null,
    null,
    null
  ],
  "Mass Function": [
    null,
    null,
    null
  ],
  "M1_sin3i": [
    null,
    null,
    null
  ],
  "M2_sin3i": [
    null,
    null,
    null
  ],
  "evol_type_1": "AGB",
  "evol_type_2": null,
  "obs_type_1": "Post-AGB",
  "obs_type_2": null,
  "system_class": "Post-AGB binary",
  "Detection Method": [
    "RV"
  ],
  "Reference": [
    "2022A&A...658A..36K"
  ],
  "Notes": "Source: Kluska+2021 table 1 (Cat. 4). Raw row:

In [47]:
# Print number of entries with nonzero eccentricity
n_ecc = sum(1 for e in merged_entries
            if has_value(e.get("Eccentricity")) and float(e["Eccentricity"][1]) > 0)
print(f"\nNumber of systems with nonzero eccentricity: {n_ecc} / {n_unique} ({n_ecc/n_unique:.1%})")    

# Print number of systems with P value <= 0
n_period = sum(1 for e in merged_entries
               if has_value(e.get("Period")) and float(e["Period"][1]) <= 0)
print(f"\nNumber of systems with negative period measurement: {n_period} / {n_unique } ({n_period/n_unique:.1%})")   


Number of systems with nonzero eccentricity: 34 / 96 (35.4%)

Number of systems with negative period measurement: 0 / 96 (0.0%)


In [ ]:
print("DEBUG: Check HD131356 specifically in name_to_keys:")
hd_keys = {k: v for k, v in name_to_keys.items() if '131356' in k}
for k in sorted(hd_keys.keys()):
    print(f"  '{k}': {hd_keys[k]}")

print("\nDEBUG: Groups with >1 entry (duplicate merges):")
dup_groups = [g for g in groups_map.values() if len(g) > 1]
print(f"  Total duplicate groups: {len(dup_groups)}")
for i, g in enumerate(dup_groups[:10]):
    catalogs_set = set(src for src, _ in g)
    print(f"  Group {i}: {g}, catalogs={catalogs_set}")

print(f"\nDEBUG: Multi_groups (≥2 catalogs): {len(multi_groups)}")
print(f"  merged_entries count: {len(merged_entries)}")




DEBUG: Check HD131356 specifically in name_to_keys:
  'ENTRA,HD131356': [('Kluska', 23)]
  'HD131356': [('Oomen', 11), ('Moltzer', 33)]

DEBUG: Groups with >1 entry (duplicate merges):
  Total duplicate groups: 52
  Group 0: [('Kluska', 0), ('Oomen', 3), ('Moltzer', 5)], catalogs={'Oomen', 'Kluska', 'Moltzer'}
  Group 1: [('Kluska', 1), ('Oomen', 31), ('Moltzer', 23)], catalogs={'Oomen', 'Kluska', 'Moltzer'}
  Group 2: [('Kluska', 2), ('Moltzer', 29)], catalogs={'Kluska', 'Moltzer'}
  Group 3: [('Kluska', 6), ('Oomen', 8), ('Moltzer', 31)], catalogs={'Oomen', 'Kluska', 'Moltzer'}
  Group 4: [('Kluska', 7), ('Oomen', 28), ('Moltzer', 2)], catalogs={'Oomen', 'Kluska', 'Moltzer'}
  Group 5: [('Kluska', 8), ('Moltzer', 39)], catalogs={'Kluska', 'Moltzer'}
  Group 6: [('Kluska', 9), ('Oomen', 19), ('Moltzer', 16)], catalogs={'Oomen', 'Kluska', 'Moltzer'}
  Group 7: [('Kluska', 10), ('Moltzer', 40)], catalogs={'Kluska', 'Moltzer'}
  Group 8: [('Kluska', 11), ('Oomen', 20)], catalogs={'Oomen'